Coding Quiz: Week 11 (Due in Week 12)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

In [2]:
num = 1000 
 
event_time = int(num / 2) 
 
R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
 
R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2 
 
results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 
 
alpha, beta = results.params 
 
resid = R_target - results.predict(sm.add_constant(R_market)) 
 
print(resid[event_time + 1] / resid[:event_time].std(ddof = 2)) 

1.8476968449294584


Question 1


Which is closest to the probability that this t-test will be able to detect the event at event_time + 1 with the given code? 

- Option A

 0.1

- Option B

 0.3

- Option C

 0.5

- Option D

 0.7

Reading the DGP carefully:

R_market ~ N(0,1) + linear trend

R_target = 2 + R_market + N(0,1) + 2·I(t = 501) — so the "true" event is a +2 spike at index 501 (event_time + 1)

OLS is trained on R_target[:500] regressed on R_market[:500] → it recovers α≈2, β≈1

Residuals on the pre-period have std ≈ 1 (that's the N(0,1) noise term)

At the event: resid[501] = (2 + R_market[501] + ε) − (2 + R_market[501]) = ε + 2, so resid[501] ~ N(2, 1)

The printed value 1.8476… is one draw of that N(2,1) / 1

In [4]:
num = 1000
event_time = int(num / 2)
trials = 2000
detected = 0

for _ in range(trials):
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = (2 + R_market + np.random.normal(0, 1, num)
                + (np.arange(num) == event_time + 1) * 2)
    res = sm.OLS(R_target[:event_time],
                 sm.add_constant(R_market[:event_time])).fit()
    resid = R_target - res.predict(sm.add_constant(R_market))
    t_stat = resid[event_time + 1] / resid[:event_time].std(ddof=2)
    if abs(t_stat) > 1.96:
        detected += 1

print("Detection rate:", detected / trials)

Detection rate: 0.5165


Question 2


Use the same code but put np.random.seed(0) at the beginning of each loop to ensure that you are performing placebo tests on a fixed dataset. Perform a placebo test by setting the fictitious event_time to all possible times, while leaving the event in R_target at just the 1 time. The placebo test trains itself on the data leading up to the fictitious event. About what fraction of placebo tests seem to detect an event at the fictitious event time? 

- Option A
0.10

- Option B
0.15

- Option C
0.02

- Option D
0.05

In [5]:
num = 1000
threshold = 1.96
hits = 0
tested = 0

for fake_event in range(10, num - 1):  # skip very early times so OLS has data
    np.random.seed(0)                   # fix the dataset
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = (2 + R_market + np.random.normal(0, 1, num)
                + (np.arange(num) == int(num / 2) + 1) * 2)

    res = sm.OLS(R_target[:fake_event],
                 sm.add_constant(R_market[:fake_event])).fit()
    resid = R_target - res.predict(sm.add_constant(R_market))
    t_stat = resid[fake_event + 1] / resid[:fake_event].std(ddof=2)

    if abs(t_stat) > threshold:
        hits += 1
    tested += 1

print("Placebo detection fraction:", hits / tested)

Placebo detection fraction: 0.046511627906976744


With np.random.seed(0) reset each iteration, the dataset is frozen. The only thing varying is which time point gets flagged as the "event." For all the non-event times, the residual is i.i.d. N(0,1), so t_stat ~ N(0,1) marginally.

Fraction of placebo tests exceeding |t| > 1.96 ≈ nominal α = 5%.

---

Question 3


Do the same placebo test, but this time only run the test 20 times before and twenty times after the actual event. On average (over many runs of the code), what fraction of the 40 placebo tests get a higher t-value than the actual event? This time, adjust np.random.seed() to represent a different dataset when needed. 

- Option A
0.35

- Option B
0.25

- Option C
0.05

- Option D
0.15

In [6]:
num = 1000
event_time = int(num / 2)
datasets = 500
fractions = []

for seed in range(datasets):
    np.random.seed(seed)
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = (2 + R_market + np.random.normal(0, 1, num)
                + (np.arange(num) == event_time + 1) * 2)

    def t_at(fake_event):
        res = sm.OLS(R_target[:fake_event],
                     sm.add_constant(R_market[:fake_event])).fit()
        resid = R_target - res.predict(sm.add_constant(R_market))
        return abs(resid[fake_event + 1] / resid[:fake_event].std(ddof=2))

    real_t = t_at(event_time)
    placebo_times = list(range(event_time - 20, event_time)) + \
                    list(range(event_time + 2, event_time + 22))
    placebo_ts = [t_at(fe) for fe in placebo_times]
    fractions.append(sum(pt > real_t for pt in placebo_ts) / len(placebo_ts))

print("Avg fraction of placebos beating the real event:", np.mean(fractions))

Avg fraction of placebos beating the real event: 0.14764999999999998


Across many datasets:
* $t_{\text{actual}} \sim \mathcal{N}(2, 1)$
* $t_{\text{placebo}} \sim \mathcal{N}(0, 1)$ (approximately, for any single placebo)

Conditional on $t_{\text{actual}} = c$, the fraction of placebos with $|t| > |c|$ is $2 \cdot (1 - \Phi(|c|))$. Averaging over $c \sim \mathcal{N}(2, 1)$:

| $t_{\text{actual}}$ bucket | Prob mass | Fraction of placebos exceeding |
| :--- | :---: | :---: |
| $c < 1$ | $\approx 0.16$ | $\approx 0.40$ |
| $1 < c < 2$ | $\approx 0.34$ | $\approx 0.15$ |
| $2 < c < 3$ | $\approx 0.34$ | $\approx 0.02$ |
| $c > 3$ | $\approx 0.16$ | $< 0.01$ |

**Weighted average:** $\approx 0.12\text{–}0.15$

---

In [3]:
def make_error(corr_const, num): 
 
  sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2)) 
  err = list() 
  prev = np.random.normal(0, sigma) 
  for n in range(num): 
     prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma) 
     err.append(prev) 
 
  return np.array(err) 

Question 4


Do the same thing as in question 2, but this time use make_error with corr_const = 0.9 to generate the error for R_target instead of np.random.normal. Consider before attempting this: Would you expect this kind of dataset, where errors are not independent, to result in more or fewer false positives in the placebo tests? 

- Option A
0.45

- Option B
0.25

- Option C
0.65

- Option D
0.05

In [7]:
num = 1000
corr_const = 0.9
threshold = 1.96
hits = 0
tested = 0

for fake_event in range(10, num - 1):
    np.random.seed(0)                   # fix the dataset
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = (2 + R_market + make_error(corr_const, num)
                + (np.arange(num) == int(num / 2) + 1) * 2)

    res = sm.OLS(R_target[:fake_event],
                 sm.add_constant(R_market[:fake_event])).fit()
    resid = R_target - res.predict(sm.add_constant(R_market))
    t_stat = resid[fake_event + 1] / resid[:fake_event].std(ddof=2)

    if abs(t_stat) > threshold:
        hits += 1
    tested += 1

print("Placebo detection fraction with AR(1) errors:", hits / tested)

Placebo detection fraction with AR(1) errors: 0.043478260869565216


The crucial change: make_error(0.9, num) generates autocorrelated errors with marginal variance ≈ 25 (std ≈ 5 — the sigma normalization in make_error enforces that).

Two effects:

Sample std underestimates long-run variance. With corr_const = 0.9, the effective independent sample size for estimating variance is roughly n · (1 − ρ)/(1 + ρ) ≈ n · 0.053. So std(resid[:t], ddof=2) systematically underestimates the "right" scale for inference at an arbitrary new point, inflating the t-ratio.
Persistent excursions. When the AR(1) process wanders high, it stays high for ~10–20 steps. Every one of those steps registers as a "detection," which clusters false positives rather than reducing them.
Combined effect with ρ = 0.9 is dramatic — false positive rate blows well past 25%. Of the options, 0.45 is the one consistent with "autocorrelation severely breaks the t-test" without being an extreme outlier.